# Session 11 Solutions: pandas Tables, Selecting and Filtering

Data Science with Python, Level 1, Stage 2: Working with Data

Try each exercise yourself before reading its solution. There is usually more than one correct answer; compare your approach with this one.

## Setup

In [1]:
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"

df = pd.read_csv(DATA_URL + "cafe_sales.csv")
df["revenue"] = df["unit_price"] * df["quantity"]

## In-Class Exercises

### In-Class Exercise 1: First Look

In [2]:
df = pd.read_csv(DATA_URL + "cafe_sales.csv")
print(df.shape)
print(df.columns.tolist())
print(df["item"].nunique())

(1000, 9)
['order_id', 'date', 'hour', 'branch', 'item', 'category', 'unit_price', 'quantity', 'payment']
12


### In-Class Exercise 2: Busy Hours

In [3]:
df = pd.read_csv(DATA_URL + "cafe_sales.csv")
print(df["hour"].value_counts().head(3))

hour
9     113
8     107
10     89
Name: count, dtype: int64


### In-Class Exercise 3: Morning Coffee

In [4]:
morning_coffee = df[(df["category"] == "Coffee") & (df["hour"] < 10)]
print(len(morning_coffee))
print(morning_coffee["item"].value_counts().head(1))

131
item
Latte    49
Name: count, dtype: int64


### In-Class Exercise 4: Busiest Weekday

In [5]:
df["date"] = pd.to_datetime(df["date"])
print(df["date"].dt.day_name().value_counts().head(1))

date
Thursday    160
Name: count, dtype: int64


## Part A: Basic

### Exercise 1: Students Table

In [6]:
students = pd.read_csv(DATA_URL + "students.csv")
print(students.shape)
print(students.head())
print(round(students["gpa"].mean(), 2))

(30, 8)
  student_id           name  age      city             major   gpa  credits  scholarship
0       S100    Basel Shami   24  Damascus          Business  3.85       18        False
1       S101  Sofia Sabbagh   18  Damascus  Computer Science  3.70       83         True
2       S102    Emma Khoury   19  Damascus          Business  2.06      104        False
3       S103  Basel Darwish   24    Beirut       Engineering  3.62       63        False
4       S104      Lina Issa   22     Amman            Design  3.35       30        False
2.99


### Exercise 2: Scholarship Students

In [7]:
students = pd.read_csv(DATA_URL + "students.csv")
mask = students["scholarship"] & (students["gpa"] >= 3.5)
students.loc[mask, ["name", "major", "gpa"]].sort_values("gpa", ascending=False)

,name,major,gpa
20,Lina Zein,Computer Science,3.88
1,Sofia Sabbagh,Computer Science,3.70


## Part B: Intermediate

### Exercise 3: Weather Summary

In [8]:
w = pd.read_csv(DATA_URL + "weather_2024.csv")
w["temp_range"] = w["temp_max"] - w["temp_min"]
print(w.sort_values("temp_range", ascending=False).head(5)[["date", "city", "temp_range"]])
print(w[w["rain_mm"] > 0]["city"].value_counts())

           date           city  temp_range
120  2024-04-30      Port City        11.0
335  2024-12-01      Port City        11.0
564  2024-07-17  Mountain Town        11.0
582  2024-08-04  Mountain Town        11.0
462  2024-04-06  Mountain Town        11.0
city
Port City        93
Mountain Town    79
Name: count, dtype: int64


### Exercise 4: Rainy Days

`idxmax()` gives the index label of the largest value, and `loc` then reads that row.

In [9]:
w = pd.read_csv(DATA_URL + "weather_2024.csv")
w["date"] = pd.to_datetime(w["date"])
mt = w[w["city"] == "Mountain Town"]
summer_rain = mt[mt["date"].dt.month.isin([7, 8]) & (mt["rain_mm"] > 5)]
print(summer_rain[["date", "rain_mm"]])
print(mt.loc[mt["temp_max"].idxmax(), ["date", "temp_max"]])

          date  rain_mm
550 2024-07-03     17.4
date        2024-07-21 00:00:00
temp_max                   32.7
Name: 568, dtype: object


## Part C: Challenge

### Exercise 5: Label the Orders

In [10]:
df = pd.read_csv(DATA_URL + "cafe_sales.csv")
df["revenue"] = df["unit_price"] * df["quantity"]
df["period"] = "morning"
df.loc[df["hour"].between(12, 16), "period"] = "afternoon"
df.loc[df["hour"] >= 17, "period"] = "evening"
for p in ["morning", "afternoon", "evening"]:
    part = df[df["period"] == p]
    print(f"{p:<10} {len(part):>4} orders {part['revenue'].sum():>9.2f}")

morning     427 orders   2367.10
afternoon   343 orders   2054.40
evening     230 orders   1301.60


## Part D: Cumulative Project: Cafe Sales Analysis

### Exercise 6: The First Cafe Report

In [11]:
df = pd.read_csv(DATA_URL + "cafe_sales.csv")
df["revenue"] = df["unit_price"] * df["quantity"]
print("Orders:", len(df), "over", df["date"].nunique(), "days")
print(f"Total revenue: {df['revenue'].sum():.2f}, per order: {df['revenue'].mean():.2f}")
print(df["item"].value_counts().head(5))
print(df["payment"].value_counts(normalize=True).round(3))
print(df.sort_values("revenue", ascending=False).head(1)[["date", "branch", "item", "quantity", "revenue"]])

Orders: 1000 over 90 days
Total revenue: 5723.10, per order: 5.72
item
Latte               151
Cappuccino          128
Espresso            114
Chicken Sandwich     97
Croissant            86
Name: count, dtype: int64
payment
Card      0.560
Cash      0.303
Mobile    0.137
Name: proportion, dtype: float64
           date  branch              item  quantity  revenue
387  2025-02-04  Campus  Chicken Sandwich         3     19.5


### Exercise 7: Comparing the Branches

In [12]:
df["date"] = pd.to_datetime(df["date"])
feb = df[(df["date"] >= "2025-02-01") & (df["date"] < "2025-03-01")]
for b in ["Downtown", "Campus", "Mall"]:
    part = feb[feb["branch"] == b]
    print(f"{b:<9} {len(part):>4} orders, revenue {part['revenue'].sum():.2f}, per order {part['revenue'].mean():.2f}")
    print("   most popular:", part["item"].value_counts().index[0])
sandwiches = feb[feb["item"].str.contains("Sandwich")]
print("Most sandwiches:", sandwiches["branch"].value_counts().index[0])

Downtown   142 orders, revenue 773.10, per order 5.44
   most popular: Latte
Campus     118 orders, revenue 683.10, per order 5.79
   most popular: Latte
Mall        82 orders, revenue 447.40, per order 5.46
   most popular: Cappuccino
Most sandwiches: Downtown


## Part E: Quiz Answers

| Question | Answer | Explanation |
| --- | --- | --- |
| 1 | B | `shape` is `(rows, columns)`. |
| 2 | B | It counts how many times each value appears. |
| 3 | B | The mask inside the brackets keeps the matching rows. |
| 4 | B | Use `&` with brackets around each condition. |
| 5 | B | One `loc` with rows and column changes the real table. |